# Quick Demo: Searching and Matching Movie Titles

Two tools you'll need for the interactive recommender:

1. **`str.contains`** — filter a DataFrame by text patterns (supports regex)
2. **`fuzzywuzzy`** — find approximate matches when the user makes typos

In [1]:
import pandas as pd
from fuzzywuzzy import fuzz, process

# --- Fake movie dataset ---
movies = pd.DataFrame({
    'movieId': range(1, 16),
    'title': [
        'The Dark Knight (2008)',
        'The Dark Knight Rises (2012)',
        'Dark City (1998)',
        'Toy Story (1995)',
        'Toy Story 2 (1999)',
        'Toy Story 3 (2010)',
        'The Lord of the Rings: The Fellowship of the Ring (2001)',
        'The Lord of the Rings: The Two Towers (2002)',
        'The Lord of the Rings: The Return of the King (2003)',
        'Pulp Fiction (1994)',
        'Kill Bill: Vol. 1 (2003)',
        'Kill Bill: Vol. 2 (2004)',
        'The Silence of the Lambs (1991)',
        'Forrest Gump (1994)',
        'The Matrix (1999)',
    ],
    'genres': [
        'Action|Crime|Drama',
        'Action|Crime|Thriller',
        'Mystery|Sci-Fi|Thriller',
        'Animation|Comedy|Family',
        'Animation|Comedy|Family',
        'Animation|Comedy|Family',
        'Adventure|Fantasy',
        'Adventure|Fantasy',
        'Adventure|Fantasy',
        'Crime|Drama',
        'Action|Crime|Thriller',
        'Action|Crime|Thriller',
        'Crime|Horror|Thriller',
        'Comedy|Drama|Romance',
        'Action|Sci-Fi',
    ]
})

movies

C:\Users\Dilet\AppData\Roaming\Python\Python39\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


,movieId,title,genres
0,1,The Dark Knight (2008),Action|Crime|Drama
1,2,The Dark Knight Rises (2012),Action|Crime|Thriller
2,3,Dark City (1998),Mystery|Sci-Fi|Thriller
3,4,Toy Story (1995),Animation|Comedy|Family
4,5,Toy Story 2 (1999),Animation|Comedy|Family
5,6,Toy Story 3 (2010),Animation|Comedy|Family
6,7,The Lord of the Rings: The Fellowship of the R...,Adventure|Fantasy
7,8,The Lord of the Rings: The Two Towers (2002),Adventure|Fantasy
8,9,The Lord of the Rings: The Return of the King ...,Adventure|Fantasy
9,10,Pulp Fiction (1994),Crime|Drama


---
## `str.contains` — filtering text

`str.contains` checks if a string **contains** a pattern. It returns True/False for each row, so you can use it to filter.

### Basic usage: find all movies with a word in the title

In [2]:
# Find all movies with "Dark" in the title
movies[movies['title'].str.contains('Dark')]

,movieId,title,genres
0,1,The Dark Knight (2008),Action|Crime|Drama
1,2,The Dark Knight Rises (2012),Action|Crime|Thriller
2,3,Dark City (1998),Mystery|Sci-Fi|Thriller


In [3]:
# Find all movies with "Toy" in the title
movies[movies['title'].str.contains('Toy')]

,movieId,title,genres
3,4,Toy Story (1995),Animation|Comedy|Family
4,5,Toy Story 2 (1999),Animation|Comedy|Family
5,6,Toy Story 3 (2010),Animation|Comedy|Family


### Case-insensitive search

In [4]:
# This finds nothing — "dark" != "Dark"
movies[movies['title'].str.contains('dark')]

,movieId,title,genres


In [5]:
# Fix: case=False ignores upper/lower case
movies[movies['title'].str.contains('dark', case=False)]

,movieId,title,genres
0,1,The Dark Knight (2008),Action|Crime|Drama
1,2,The Dark Knight Rises (2012),Action|Crime|Thriller
2,3,Dark City (1998),Mystery|Sci-Fi|Thriller


### Filtering by genre

In [6]:
# Find all Action movies
movies[movies['genres'].str.contains('Action')]

,movieId,title,genres
0,1,The Dark Knight (2008),Action|Crime|Drama
1,2,The Dark Knight Rises (2012),Action|Crime|Thriller
10,11,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
11,12,Kill Bill: Vol. 2 (2004),Action|Crime|Thriller
14,15,The Matrix (1999),Action|Sci-Fi


In [7]:
# Find all Comedy movies
movies[movies['genres'].str.contains('Comedy')]

,movieId,title,genres
3,4,Toy Story (1995),Animation|Comedy|Family
4,5,Toy Story 2 (1999),Animation|Comedy|Family
5,6,Toy Story 3 (2010),Animation|Comedy|Family
13,14,Forrest Gump (1994),Comedy|Drama|Romance


In [8]:
# Find movies that are BOTH Action AND Sci-Fi
mask = movies['genres'].str.contains('Action') & movies['genres'].str.contains('Sci-Fi')
movies[mask]

,movieId,title,genres
14,15,The Matrix (1999),Action|Sci-Fi


### Regex: more powerful patterns

`str.contains` supports **regex** (regular expressions) by default. A few useful patterns:

In [9]:
# | means OR: find movies with "Dark" OR "Matrix" in the title
movies[movies['title'].str.contains('Dark|Matrix')]

,movieId,title,genres
0,1,The Dark Knight (2008),Action|Crime|Drama
1,2,The Dark Knight Rises (2012),Action|Crime|Thriller
2,3,Dark City (1998),Mystery|Sci-Fi|Thriller
14,15,The Matrix (1999),Action|Sci-Fi


In [10]:
# ^ means "starts with" (inside regex)
# Find titles that start with "The"
movies[movies['title'].str.contains('^The')]

,movieId,title,genres
0,1,The Dark Knight (2008),Action|Crime|Drama
1,2,The Dark Knight Rises (2012),Action|Crime|Thriller
6,7,The Lord of the Rings: The Fellowship of the R...,Adventure|Fantasy
7,8,The Lord of the Rings: The Two Towers (2002),Adventure|Fantasy
8,9,The Lord of the Rings: The Return of the King ...,Adventure|Fantasy
12,13,The Silence of the Lambs (1991),Crime|Horror|Thriller
14,15,The Matrix (1999),Action|Sci-Fi


In [11]:
# \d means "any digit"
# Find titles that contain a year in the 2000s
movies[movies['title'].str.contains('200\d')]

,movieId,title,genres
0,1,The Dark Knight (2008),Action|Crime|Drama
6,7,The Lord of the Rings: The Fellowship of the R...,Adventure|Fantasy
7,8,The Lord of the Rings: The Two Towers (2002),Adventure|Fantasy
8,9,The Lord of the Rings: The Return of the King ...,Adventure|Fantasy
10,11,Kill Bill: Vol. 1 (2003),Action|Crime|Thriller
11,12,Kill Bill: Vol. 2 (2004),Action|Crime|Thriller


In [12]:
# .* means "anything in between"
# Find titles with "Lord" followed eventually by "Ring"
movies[movies['title'].str.contains('Lord.*Ring')]

,movieId,title,genres
6,7,The Lord of the Rings: The Fellowship of the R...,Adventure|Fantasy
7,8,The Lord of the Rings: The Two Towers (2002),Adventure|Fantasy
8,9,The Lord of the Rings: The Return of the King ...,Adventure|Fantasy


In [13]:
# Combine: genres containing Thriller but NOT Action
mask = movies['genres'].str.contains('Thriller') & ~movies['genres'].str.contains('Action')
movies[mask]

,movieId,title,genres
2,3,Dark City (1998),Mystery|Sci-Fi|Thriller
12,13,The Silence of the Lambs (1991),Crime|Horror|Thriller


### Quick reference

| Pattern | Meaning | Example |
|---|---|---|
| `'Dark'` | Contains "Dark" exactly | `str.contains('Dark')` |
| `case=False` | Ignore case | `str.contains('dark', case=False)` |
| `'Action\|Comedy'` | Contains Action OR Comedy | `str.contains('Action\|Comedy')` |
| `'^The'` | Starts with "The" | `str.contains('^The')` |
| `'\(199\d\)'` | Year in the 1990s | `str.contains('\(199\d\)')` |
| `'Lord.*Ring'` | "Lord" then anything then "Ring" | `str.contains('Lord.*Ring')` |
| `~` | NOT (negate) | `~str.contains('Horror')` |

**Tip:** if your search string has special regex characters (like parentheses or dots), use `regex=False` to treat it as plain text: `str.contains('Vol. 1', regex=False)`

---
## `fuzzywuzzy` — handling typos and approximate matches

Users won't type exact titles. They'll write `"the dark night"` instead of `"The Dark Knight (2008)"`. Fuzzy matching finds the closest match.

### `fuzz.ratio` — basic similarity score (0–100)

In [14]:
# Exact match = 100
print(fuzz.ratio('The Dark Knight', 'The Dark Knight'))

# Close match
print(fuzz.ratio('the dark night', 'The Dark Knight'))

# Partial overlap
print(fuzz.ratio('dark', 'The Dark Knight'))

# Totally different
print(fuzz.ratio('Toy Story', 'The Dark Knight'))

100
83
32
25


### `fuzz.partial_ratio` — matches substrings

Better when the user types only part of the title:

In [15]:
# "dark knight" is fully contained in the full title → high score
print(fuzz.partial_ratio('dark knight', 'The Dark Knight (2008)'))

# "toy story" matches even with the year
print(fuzz.partial_ratio('toy story', 'Toy Story 2 (1999)'))

# Compare: ratio vs partial_ratio
print(f"ratio:         {fuzz.ratio('toy story', 'Toy Story 2 (1999)')}")
print(f"partial_ratio: {fuzz.partial_ratio('toy story', 'Toy Story 2 (1999)')}")

82
78
ratio:         52
partial_ratio: 78


### `fuzz.token_sort_ratio` — ignores word order

Useful when users swap words around:

In [16]:
# "knight dark the" → same words, different order
print(fuzz.token_sort_ratio('knight dark the', 'The Dark Knight'))

# Works with extra words too
print(fuzz.token_sort_ratio('rings lord of the', 
                             'The Lord of the Rings: The Fellowship of the Ring (2001)'))

100
49


### `process.extractBests` — find the best matches in a list

This is what you'll actually use in the recommender: give it a query and a list of titles, and it returns the top matches.

In [17]:
# User types "dark night" — find best matches
titles = movies['title'].tolist()

results = process.extractBests('dark night', titles, limit=5, score_cutoff=50)

print("User typed: 'dark night'")
print(f"{'Match':<55s} {'Score'}")
print("-" * 65)
for title, score in results:
    print(f"{title:<55s} {score}")

User typed: 'dark night'
Match                                                   Score
-----------------------------------------------------------------
The Dark Knight (2008)                                  86
The Dark Knight Rises (2012)                            86
Dark City (1998)                                        86


In [18]:
# User types "toy store" (typo!)
results = process.extractBests('toy store', titles, limit=5, score_cutoff=50)

print("User typed: 'toy store'")
print(f"{'Match':<55s} {'Score'}")
print("-" * 65)
for title, score in results:
    print(f"{title:<55s} {score}")

User typed: 'toy store'
Match                                                   Score
-----------------------------------------------------------------
Toy Story (1995)                                        86
Toy Story 2 (1999)                                      86
Toy Story 3 (2010)                                      86
The Lord of the Rings: The Two Towers (2002)            57


In [19]:
# User types "lord of the rings"
results = process.extractBests('lord of the rings', titles, limit=5, score_cutoff=50)

print("User typed: 'lord of the rings'")
print(f"{'Match':<55s} {'Score'}")
print("-" * 65)
for title, score in results:
    print(f"{title:<55s} {score}")

User typed: 'lord of the rings'
Match                                                   Score
-----------------------------------------------------------------
The Lord of the Rings: The Fellowship of the Ring (2001) 90
The Lord of the Rings: The Two Towers (2002)            90
The Lord of the Rings: The Return of the King (2003)    90
The Dark Knight Rises (2012)                            86
The Silence of the Lambs (1991)                         86


In [20]:
# User types something completely wrong
results = process.extractBests('star wars', titles, limit=5, score_cutoff=50)

print("User typed: 'star wars'")
if results:
    for title, score in results:
        print(f"  {title} (score={score})")
else:
    print("  No match found above threshold!")

User typed: 'star wars'
  No match found above threshold!


### Putting it together: a simple match function

In [21]:
def match_movie(user_input, movie_titles, threshold=60):
    """
    Match user input to the closest movie title.
    Returns (matched_title, score) or None if no good match.
    """
    results = process.extractBests(user_input, movie_titles, 
                                    limit=3, score_cutoff=threshold)
    
    if not results:
        return None
    
    best_title, best_score = results[0]
    
    print(f"  Best match: '{best_title}' (score={best_score})")
    if len(results) > 1:
        print(f"  Other options:")
        for title, score in results[1:]:
            print(f"    - '{title}' (score={score})")
    
    return best_title, best_score

# Test it
print("Input: 'pulp fiction'")
match_movie('pulp fiction', titles)
print()
print("Input: 'silence of lambs'")
match_movie('silence of lambs', titles)
print()
print("Input: 'forest gum'")
match_movie('forest gum', titles)
print()
print("Input: 'avengers'")
match_movie('avengers', titles, threshold=50)

Input: 'pulp fiction'
  Best match: 'Pulp Fiction (1994)' (score=90)

Input: 'silence of lambs'
  Best match: 'The Lord of the Rings: The Fellowship of the Ring (2001)' (score=86)
  Other options:
    - 'The Lord of the Rings: The Two Towers (2002)' (score=86)
    - 'The Lord of the Rings: The Return of the King (2003)' (score=86)

Input: 'forest gum'
  Best match: 'Forrest Gump (1994)' (score=81)

Input: 'avengers'
